<a href="https://colab.research.google.com/github/Ayushjha02/speech-to-text/blob/main/F_speechtoText.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd
audio_file_metadata = pd.read_csv("/content/drive/MyDrive/metadata.csv")

In [6]:
audio_file_metadata.head()

,Unnamed: 0,id,sentence,file_name,audio_path
0,0,LJ001-0001,"Printing, in the only sense with which we are ...",LJ001-0001.wav,/kaggle/input/ljspeech-dataset/LJSpeech-1.1/wa...
1,1,LJ001-0002,in being comparatively modern.,LJ001-0002.wav,/kaggle/input/ljspeech-dataset/LJSpeech-1.1/wa...
2,2,LJ001-0003,For although the Chinese took impressions from...,LJ001-0003.wav,/kaggle/input/ljspeech-dataset/LJSpeech-1.1/wa...
3,3,LJ001-0004,"produced the block books, which were the immed...",LJ001-0004.wav,/kaggle/input/ljspeech-dataset/LJSpeech-1.1/wa...
4,4,LJ001-0005,the invention of movable metal letters in the ...,LJ001-0005.wav,/kaggle/input/ljspeech-dataset/LJSpeech-1.1/wa...


#**keeping required rows**

In [7]:
df = audio_file_metadata[['file_name','sentence']]
df = df.rename(columns={
    "file_name": "audio",
    "sentence": "text"
})



In [8]:
df.head()

,audio,text
0,LJ001-0001.wav,"Printing, in the only sense with which we are ..."
1,LJ001-0002.wav,in being comparatively modern.
2,LJ001-0003.wav,For although the Chinese took impressions from...
3,LJ001-0004.wav,"produced the block books, which were the immed..."
4,LJ001-0005.wav,the invention of movable metal letters in the ...


In [95]:
import os
for value in df['audio']:
 audio_path = '/content/drive/MyDrive/wavs (1)/' + value
 if os.path.exists(audio_path):
  continue
 else:
  df.drop(df[df['audio'] == value].index, inplace = True)


In [96]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2160 entries, 0 to 2159
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   audio   2160 non-null   object
 1   text    2160 non-null   object
dtypes: object(2)
memory usage: 50.6+ KB


**Read file from the folder**

In [1]:
!pip install --upgrade --force-reinstall librosa

  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached audioread-3.1.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached numba-0.65.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.9 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached decorator-5.2.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached soundfile-0.13.1-py2.py3-none-manylinux_2_28_x86_64.whl.metadata (16 kB)
  Using cached pooch-1.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached soxr-1.1.0-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metad

In [97]:
import torch
from torch.utils.data import Dataset
import librosa
import os


In [98]:
import string

class CharTokenizer:
    def __init__(self):
        # vocabulary
        self.chars = list(string.ascii_lowercase + " '")
        self.chars.append("_")  # blank token for CTC

        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.idx_to_char = {i: c for c, i in self.char_to_idx.items()}

        self.blank_idx = self.char_to_idx["_"]

    # text → tokens
    def encode(self, text):
        text = text.lower()
        return [self.char_to_idx[c] for c in text if c in self.char_to_idx]

    # tokens → text
    def decode(self, indices):
        return "".join([self.idx_to_char[i] for i in indices])

    # CTC decoding (remove duplicates + blanks)
    def ctc_decode(self, indices):
        decoded = []
        prev = None

        for i in indices:
            if i != prev and i != self.blank_idx:
                decoded.append(self.idx_to_char[i])
            prev = i

        return "".join(decoded)





In [99]:
import os

print(len(os.listdir('/content/drive/MyDrive/wavs (1)')))

2166


In [100]:
from numpy import char
import torch
from torch.utils.data import Dataset
import librosa
import numpy as np
import os
import re
from pathlib import Path



def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z' ]", "", text)
    return text
class STTDataset(Dataset,CharTokenizer):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
        super().__init__()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        audio_path = '/content/drive/MyDrive/wavs (1)/' + row["audio"]
        text = clean_text(row["text"])


        file_path = Path(audio_path)

        if file_path.exists():
            if file_path != None:
             audio, sr = librosa.load(file_path, sr=16000)

        # Convert to spectrogram
             mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=80)
             mel = torch.tensor(mel.T, dtype=torch.float32)


        # Encode text
            tokens = torch.tensor(self.encode(text=text), dtype=torch.long)

            return mel, tokens

In [101]:
dataset = STTDataset(df)

In [105]:
def collate_fn(batch):
    # Filter out None values from the batch
    batch = [sample for sample in batch if sample is not None]

    # If the whole batch is empty, return None
    if len(batch) == 0:
        return None

    # Separate inputs (mel spectrograms) and targets (tokens)
    mels, tokens = zip(*batch)

    # Pad mel spectrograms to the longest in the batch
    max_mel_len = max(mel.shape[0] for mel in mels)
    padded_mels = [
        torch.nn.functional.pad(mel, (0, 0, 0, max_mel_len - mel.shape[0]), 'constant', 0)
        for mel in mels
    ]
    padded_mels = torch.stack(padded_mels)

    # Pad tokens to the longest in the batch
    max_token_len = max(token.shape[0] for token in tokens)
    padded_tokens = [
        torch.nn.functional.pad(token, (0, max_token_len - token.shape[0]), 'constant', 0)
        for token in tokens
    ]
    padded_tokens = torch.stack(padded_tokens)

    # Create mel_lens and token_lens tensors for sequence lengths
    mel_lens = torch.tensor([mel.shape[0] for mel in mels], dtype=torch.long)
    token_lens = torch.tensor([token.shape[0] for token in tokens], dtype=torch.long)

    return padded_mels, padded_tokens, mel_lens, token_lens

In [113]:
from torch.utils.data import DataLoader
if dataset is not None:
 loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

tensor([[[5.6201e-04, 2.5121e-03, 2.5035e-03,  ..., 2.1046e-06,
          1.0930e-05, 3.6760e-06],
         [1.6759e-03, 6.5834e-03, 1.1522e-02,  ..., 1.2740e-04,
          3.9420e-04, 7.6264e-05],
         [1.0693e-03, 2.4123e-03, 1.2074e-02,  ..., 6.6874e-04,
          1.5155e-03, 3.0075e-04],
         ...,
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00]],

        [[2.1238e-05, 1.9612e-04, 7.9936e-04,  ..., 2.1486e-01,
          1.8561e-01, 3.7298e-02],
         [2.0020e-05, 1.3014e-04, 1.3041e-03,  ..., 2.6041e+00,
          1.5731e+00, 4.4501e-01],
         [2.5542e-05, 5.7510e-05, 1.8425e-03,  ..., 7.7360e+00,
          3.2956e+00, 7.4210e-01],
         ...,
         [4.8903e-06, 1.6864e-04, 4.1604e-03,  ..., 7.6105e-07,
          6.600

KeyboardInterrupt: 